In [ ]:
import scanpy as sc
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
adata = sc.read_h5ad(data_dir / "processed" / "00_combined_processed.h5ad")
immune = adata[adata.obs["lineage"] == "immune"].copy()

In [ ]:
immune = deviant_hvg(immune)

In [ ]:
plot_deviance(immune)

In [ ]:
harmony_embed(immune)
sc.pl.pca_overview(immune, color="sample_id")

In [ ]:
cluster_subset(immune, n_pcs=30, resolution=1.5, min_dist=1)

In [ ]:
sc.pl.umap(immune, color=["scDblFinder_score", "sample_id", "pct_counts_mt", "leiden"], ncols=2)

In [ ]:
immune_markers = {
    "alv_macs": ["Siglecf", "Marco", "Pparg"],
    "inter_macs": ["C1qa", "C1qb", "Folr2"],
    "monocytes": ["Ccr2", "Plac8", "Ace", "Adgre4", "Cd300e", "Nr4a1", "Fn1"],
    "dcs": ["Flt3", "Zbtb46", "Itgax", "Ccr7", "Fscn1", "Siglech", "Bst2", "Ccr9"],
    "neutrophils": ["S100a8", "Asprv1", "Retnlg", "Ly6g"],
    "nks/ilcs": ["Ncr1", "Klrb1c", "Eomes", "Gata3", "Rorc", "Il1rl1", "Areg", "Il7r", "Il22"],
    "b cells": ["Cd19", "Cd79a"],
    "t cells": ["Cd3e", "Cd4", "Cd8a"],
    "mast cells": ["Cpa3", "Kit"],
    "megakaryocytes": ["Tubb1", "Vwf", "Ppbp"],
    "basophils": ["Cd200r3"],
    "proliferating cells": ["Mki67", "Top2a"]
}

sc.tl.dendrogram(immune, groupby="leiden", use_rep="X_pca_harmony")
sc.pl.matrixplot(
    immune,
    var_names=immune_markers,
    groupby="leiden",
    standard_scale="var",
    dendrogram=True,
    cmap="Reds")

In [ ]:
print("Before removing: " + str(immune.shape[0]))
immune = immune[immune.obs["leiden"] != "7"].copy()
print("After removing: " + str(immune.shape[0]))

In [ ]:
immune_celltypes = {
    "alv_macs": ["8", "15", "22", "9", "0", "10", "20", "14"],
    "prolif alv_macs": ["16", "4"],
    "inter_macs": ["12", "11"],
    "prolif inter_macs": ["23"],
    "nonclassical monocytes": ["13"],
    "classical monocytes": ["18"],
    "pdcs": ["25"],
    "migratory dcs": ["26"],
    "cdcs": ["17"],
    "neutrophils": ["21"],
    "nks/ilcs": ["2"],
    "b cells": ["5", "6"],
    "prolif b cells": ["1"],
    "t cells": ["3"],
    "mast cells/megakaryocytes": ["24"],
    "basophils": ["19"]
}

apply_celltypes(immune, immune_celltypes)

In [ ]:
sc.pl.umap(immune, color="celltype", legend_loc="on data", legend_fontsize=6)

In [ ]:
immune.write_h5ad(data_dir / "processed" / "01_immune.h5ad")